# Alfa-Code — Colab Free T4 (15GB)
Runtime → Change runtime type → T4 GPU. Then Run all.

📦 Default `code-100k` (1.5B code+reasoning, deploy 100K ctx via YaRN):
- Colab-এ **seq 4096 + grad-accum 16** (single T4 VRAM-safe; effective batch 16 = Kaggle-এর সমান)
- **Full train (8192 FSDP) = Kaggle `kaggle_t4.ipynb` (2×T4)** — এখানে short run / demo
- Free tier ~6h session: ROWS 10000 কম হলে বা OOM হলে `SEQ_LEN = 2048` (cell 5) করে দিন
- Crash/disconnect হলে cell 7-এর resume line চালান (`--resume-from {OUT}`)

শেষে: upload cell → **100K-context demo cell** → UI।


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q torch transformers datasets tokenizers accelerate huggingface_hub pyyaml bitsandbytes

In [ ]:
import torch
print('torch:', torch.__version__, '| cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
else:
    print('WARNING: GPU not visible to torch. Runtime -> Change runtime type -> T4 GPU, then Runtime -> Restart session and Run all.')


In [ ]:
from huggingface_hub import login
from google.colab import userdata
try:
    tok = userdata.get('HF_TOKEN')
except Exception:
    tok = None
if tok:
    login(token=tok)
    print('logged in')
else:
    print('No HF_TOKEN secret — Hub upload step will be skipped. Add it via the key icon if needed.')


In [ ]:
# Code comes from public GitHub (no token needed) — avoids 401 from private HF repo
%cd /content
!rm -rf /content/alfa
!git clone https://github.com/mrsandipmandal/alfa-code.git /content/alfa
%cd /content/alfa
!ls scripts/ configs/

In [ ]:
# Pick model ONCE here — every cell below follows it.
MODEL = "code-100k"  # 1.5B code+reasoning (deploy 100K ctx via YaRN)
# other options: "base-1B", "tiny-50M"
# ACCUM: single T4 VRAM safety (effective batch = BS * ACCUM must stay ~16)
if MODEL == "code-100k":
    # single T4: seq 4096 here; the full 8192 FSDP run lives in kaggle_t4.ipynb
    CONFIG, OUT, EPOCHS, SEQ_LEN, BS, ROWS, CHAT_ROWS = (
        "configs/code-100k.yaml", "/content/outputs/code-100k", 1, 4096, 1, 10000, 0)
    ACCUM = 16
elif MODEL == "base-1B":
    CONFIG, OUT, EPOCHS, SEQ_LEN, BS, ROWS, CHAT_ROWS = (
        "configs/base-1B.yaml", "/content/outputs/base-1B", 3, 1024, 1, 5000, 2000)
    ACCUM = 16
else:
    CONFIG, OUT, EPOCHS, SEQ_LEN, BS, ROWS, CHAT_ROWS = (
        "configs/tiny-50M.yaml", "/content/outputs/tiny-50M", 1, 2048, 2, 1500, 500)
    ACCUM = 8
print("MODEL:", MODEL, "| OUT:", OUT, "| epochs:", EPOCHS, "| seq:", SEQ_LEN,
      "| bs:", BS, "| accum:", ACCUM, "| eff_batch:", BS * ACCUM, "| rows:", ROWS)


In [ ]:
if MODEL == "code-100k":
    !python scripts/download_data.py --dataset code_mix --max-rows {ROWS} --overwrite
else:
    !python scripts/download_data.py --dataset multimodal_mix --max-rows {ROWS} --overwrite
    !python scripts/download_data.py --dataset chat_qa --max-rows {CHAT_ROWS}
!python scripts/train_tokenizer.py
!ls -lh tokenizers/ data/processed/


In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"  # less fragmentation OOM
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
# TRAIN (single T4): adamw_8bit from config (bitsandbytes installed above),
# grad-accum ACCUM keeps VRAM safe; save every 100 steps (weights-only).
# OOM at 4096? set SEQ_LEN = 2048 in cell 5 and rerun from here.
# Crash/disconnect? add  --resume-from {OUT}  to the line below and rerun.
!python scripts/train_hf.py --data data/processed/train.jsonl --out {OUT} --epochs {EPOCHS} --config {CONFIG} --max-seq-len {SEQ_LEN} --batch-size {BS} --grad-accum {ACCUM} --save-steps 100 --save-total-limit 1
!ls -lh {OUT}


In [ ]:
# SELF-LEARNING CYCLE (Colab manual): generate -> filter -> append -> retrain.
# Bump CYCLE every round: 1, 2, 3... Review data before promoting.
CYCLE = 1
!python scripts/self_learn.py --cycle {CYCLE} --ckpt {OUT} --topics data/seeds/topics.txt --from-train 100 --n 200 --max-tokens 256 --append
# retrain on merged data into a VERSIONED folder (does NOT overwrite your model):
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
!python scripts/train_hf.py --data data/processed/train.jsonl --out /content/outputs/self-cycle-{CYCLE} --epochs 1 --resume-from {OUT} --config {CONFIG} --max-seq-len {SEQ_LEN} --batch-size {BS} --grad-accum {ACCUM} --save-steps 100 --save-total-limit 1
!ls -lh /content/outputs/self-cycle-{CYCLE}
# review (infer sample + cycle report), then promote when happy:
#   !cp -r /content/outputs/self-cycle-{CYCLE} {OUT}
# then run the upload cell below.


In [ ]:
# Push back to Hub (uploads every trained model found: code-100k, base-1B, self-cycle-N).
# Needs HF_TOKEN in Colab Secrets: key icon on the left.
import os
from pathlib import Path
tok = None
try:
    from google.colab import userdata
    tok = userdata.get('HF_TOKEN')
except Exception:
    tok = os.getenv('HF_TOKEN')
if not tok:
    print('SKIP: no HF_TOKEN secret set.')
else:
    from huggingface_hub import HfApi
    api = HfApi(token=tok)
    found = sorted(p for p in Path('/content/outputs').glob('*') if (p / 'config.json').exists())
    if not found:
        print('nothing trained yet in /content/outputs')
    for local in found:
        api.upload_folder(folder_path=str(local), repo_id='mrsandip/Alfa-Code', path_in_repo=f'outputs/{local.name}')
        print('uploaded', local.name)


In [ ]:
# 100K-context inference demo (code-100k: trained @4096/8192, YaRN -> 100000).
# Uses local OUT if trained, else pulls outputs/code-100k from the private Hub.
# T4-safe: fp16 weights ~3GB + KV grows only with the actual prompt length.
import os
from pathlib import Path
CKPT = OUT
if not (Path(CKPT) / "config.json").exists():
    try:
        from huggingface_hub import snapshot_download
        d = snapshot_download("mrsandip/Alfa-Code", allow_patterns=["outputs/code-100k/*"])
        CKPT = os.path.join(d, "outputs/code-100k")
    except Exception as e:
        raise SystemExit(
            "code-100k এখনো নেই — train cell (7) চালান, অথবা Kaggle থেকে upload করুন, "
            "তারপর এই cell আবার চালান। (Hub pull failed: %s)" % e)
print("ckpt:", CKPT)
# Long prompt then continue the code — expect coherent long-range use.
!python scripts/infer.py --local {CKPT} --context 100000 --prompt "# full app.py\nfrom flask import Flask, request, jsonify\napp = Flask(__name__)\n" --wrap --max-new-tokens 300 --max-prompt-tokens 4096


In [ ]:
# Launch UI: pulls latest code, then serves the newest trained model below (else mock).
# Open the gradio.live link that appears below.
%cd /content/alfa
!git pull origin main
import os
from pathlib import Path
for _cand in ['/content/outputs/code-100k', '/content/outputs/base-1B', '/content/outputs/tiny-50M']:
    if (Path(_cand) / 'config.json').exists():
        os.environ['ALFA_MODEL'] = _cand
        print('using model:', _cand)
        break
import importlib, app.model_backend as _mb, app.ui as _ui
importlib.reload(_mb); importlib.reload(_ui)
_ui.launch_share()
